In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
from wordcloud import WordCloud
from nltk.corpus import opinion_lexicon
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import numpy as np

# 1. Import data.

In [6]:
csv_paths = list(Path("../Dataset/data_new_approach/processed_data/").glob("*.csv"))

print("CSV files found:")
for p in csv_paths:
    print(p.name)

CSV files found:
new_preprocessed_lexicon.csv


In [8]:
file_path = csv_paths[0]
lexicon_df = pd.read_csv(file_path)
lexicon_df.head()

,platform,storefront,app_id,review_id,date,user,rating,title,review,version,package,thumbsUpCount,appVersion,source_file,raw_text,raw_length_words,bank,language
0,iOS,us,1.489512e+09,13976661175,2026-04-19 18:13:04-07:00,errttrwfdsbvxd,3,No ATM option,No debit card 😣,2.20.0,NaN,NaN,NaN,marcus_by_goldman_sachs__combined_us.csv,No ATM option No debit card 😣,7,Marcus By Goldman Sachs,en
1,iOS,us,1.489512e+09,13968552342,2026-04-17 13:06:01-07:00,Chi town $&@&$,5,Great Bank.,Really easy bank to use. The best interest rat...,2.20.0,NaN,NaN,NaN,marcus_by_goldman_sachs__combined_us.csv,Great Bank. Really easy bank to use. The best ...,13,Marcus By Goldman Sachs,en
2,iOS,us,1.489512e+09,13965458699,2026-04-16 16:11:11-07:00,Ann22k,5,Saving or CD,Easy to navigate and transfers back to account...,2.20.0,NaN,NaN,NaN,marcus_by_goldman_sachs__combined_us.csv,Saving or CD Easy to navigate and transfers ba...,18,Marcus By Goldman Sachs,en
3,iOS,us,1.489512e+09,13963956841,2026-04-16 06:48:48-07:00,binbincheong,5,Easy peasy,Very easy to use and efficient! Good interest...,2.20.0,NaN,NaN,NaN,marcus_by_goldman_sachs__combined_us.csv,Easy peasy Very easy to use and efficient! Go...,11,Marcus By Goldman Sachs,en
4,iOS,us,1.489512e+09,13959051073,2026-04-14 19:38:27-07:00,tony ballognie,3,Needs widgets,I love the interface and how simple it is BUT ...,2.20.0,NaN,NaN,NaN,marcus_by_goldman_sachs__combined_us.csv,Needs widgets I love the interface and how sim...,25,Marcus By Goldman Sachs,en


# 2. Data Overview

In [9]:
lexicon_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4478 entries, 0 to 4477
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   platform          4478 non-null   object 
 1   storefront        4478 non-null   object 
 2   app_id            2118 non-null   float64
 3   review_id         4478 non-null   object 
 4   date              2118 non-null   object 
 5   user              4478 non-null   object 
 6   rating            4478 non-null   int64  
 7   title             2118 non-null   object 
 8   review            4478 non-null   object 
 9   version           2118 non-null   object 
 10  package           2360 non-null   object 
 11  thumbsUpCount     2360 non-null   float64
 12  appVersion        2181 non-null   object 
 13  source_file       4478 non-null   object 
 14  raw_text          4478 non-null   object 
 15  raw_length_words  4478 non-null   int64  
 16  bank              4478 non-null   object 


In [10]:
# create a custom stopword list
custom_words = pd.DataFrame({
    "word": ["app", "bank", "banking", "mobile", "iphone", "apple", "android", "phone"],
    "lexicon": "custom"
})

default_stop_words = pd.DataFrame({
    "word": list(ENGLISH_STOP_WORDS),
    "lexicon": "sklearn"
})

custom_stop_words = pd.concat(
    [custom_words, default_stop_words],
    ignore_index=True
).drop_duplicates(subset=["word"])

# 3. Unigram Sentiment Analysis

## BING